# FedLegal-style independent-task baseline on the Indian High Court data

Runs the friend's baseline (case type and disposal only, 3 seeds each, single-task FedAvg, InLegalBERT + LoRA, 256 head+tail tokens) on the **same split, masking and labels** as the main model, with two fixes so its numbers are comparable:
1. case-type labels are mapped to the baseline's names, so no cases are dropped (before, 81 motor-accident cases were lost);
2. case-type macro-F1 is computed over the classes present, as in the main model (before, every family missing from a court's test set counted as F1 = 0), and the duration output starts at log(90 days), as in the main model.

**Inputs:** the dataset with the repo folder (`fl_judicial.py` + `FedLegal_India_Three_Task_Baseline/`) and the dataset with `judgments_labelled.csv`. **Settings:** GPU T4 x2, Internet On. Use **Save & Run All (Commit)** so it is not interrupted.

In [ ]:
import glob, os, shutil, subprocess, sys, time
!pip uninstall -y -q torchao
!pip -q install "transformers>=4.35,<5" peft accelerate
# find the repo (folder containing both fl_judicial.py and the baseline) and the data, wherever they are attached
repos = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/fl_judicial.py", recursive=True)
         if os.path.isdir(os.path.join(os.path.dirname(p), "FedLegal_India_Three_Task_Baseline"))]
assert repos, "no attached folder contains both fl_judicial.py and FedLegal_India_Three_Task_Baseline/"
csvs = [p for p in glob.glob("/kaggle/input/**/*.csv", recursive=True) if "judgments_labelled" in os.path.basename(p)]
assert csvs, "judgments_labelled.csv not found under /kaggle/input"
ROOT, CSV = "/kaggle/working/proj", csvs[0]
shutil.rmtree(ROOT, ignore_errors=True); shutil.copytree(repos[0], ROOT)    # fresh, writable copy every run
BASE = ROOT + "/FedLegal_India_Three_Task_Baseline"
print("repo:", repos[0], "| data:", CSV)

In [ ]:
%%writefile /kaggle/working/make_adapter_csv.py
"""judgments_labelled.csv -> the baseline's columns, with the main repo's masking and label names."""
import sys, pandas as pd
sys.path.insert(0, sys.argv[1])            # repo root containing fl_judicial.py
from fl_judicial import mask_text
src, dst = sys.argv[2], sys.argv[3]
df = pd.read_csv(src, low_memory=False)
if "garbled" in df: df = df[~df.garbled.astype(bool)]
TYPE = {"WRIT": "writ", "BAIL": "bail", "CIVIL_APPEAL_OR_SUIT": "civil", "CRIMINAL_OTHER": "criminal",
        "OTHER": "other", "REVIEW_REVISION": "review", "CONTEMPT": "contempt", "ARBITRATION": "arbitration",
        "MOTOR_ACCIDENT": "motor accident", "TAX": "tax"}
ct = df.case_type_group.fillna("OTHER").map(TYPE)
assert ct.notna().all(), f"unmapped case types: {sorted(df.case_type_group[ct.isna()].unique())}"
out = pd.DataFrame({
    "court": df.court, "split": df.split,
    "text": df.input_text.fillna("").astype(str).map(mask_text),     # same masking as fl_judicial.py
    "duration_days": df.duration_days,
    "case_type": ct,
    "disposal": df.outcome_from_text.astype(str).str.lower(),       # 'unclear' is dropped for disposal only
})
out.to_csv(dst, index=False)
print(f"{len(df)} rows -> {dst}"); print(out.case_type.value_counts().to_string())


In [ ]:
%%writefile /kaggle/working/patch_baseline.py
"""Align the baseline's metric and initialisation with fl_judicial.py (run once per fresh copy)."""
import sys
base = sys.argv[1]
def patch(path, old, new):
    s = open(path).read()
    if new in s: print("already patched:", path); return
    assert old in s, f"pattern not found in {path}"
    open(path, "w").write(s.replace(old, new)); print("patched:", path)
# 1) case-type macro-F1 over the classes present (as in fl_judicial.py), not over all 10:
#    otherwise every family missing from a court's test set counts as F1 = 0
patch(f"{base}/fedlegal_india/metrics.py",
      "labels=list(range(10 if task=='case_type' else 4)),zero_division=0)",
      "labels=(None if task=='case_type' else [0,1,2,3]),zero_division=0)")
# 2) start the duration output near log(90 days), as fl_judicial.py does
patch(f"{base}/fedlegal_india/model.py",
      "        self.head = nn.Linear(hidden, 1 if task == 'duration' else num_labels)\n",
      "        self.head = nn.Linear(hidden, 1 if task == 'duration' else num_labels)\n"
      "        if task == 'duration': nn.init.constant_(self.head.bias, 4.5)\n")


In [ ]:
!python /kaggle/working/patch_baseline.py {BASE}
!python /kaggle/working/make_adapter_csv.py {ROOT} "{CSV}" /kaggle/working/adapter.csv

In [ ]:
# 6 runs (2 tasks x 3 seeds) split over both GPUs; each GPU runs its jobs one after another
OUT = "/kaggle/working/outputs"
TASKS = ("case_type", "disposal")          # duration not run
jobs = [(t, s) for s in (0, 1, 2) for t in TASKS]
def queue(gpu, my_jobs):
    cmds = " ; ".join(f"python -m fedlegal_india.train --data /kaggle/working/adapter.csv --task {t} --seed {s} "
                       f"--split-col split --text-already-masked --output {OUT}" for t, s in my_jobs)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    return subprocess.Popen(cmds, shell=True, cwd=BASE, env=env, stdout=open(f"/kaggle/working/gpu{gpu}.log", "w"), stderr=subprocess.STDOUT)
import torch
n = max(1, min(2, torch.cuda.device_count()))
procs = [queue(g, jobs[g::n]) for g in range(n)]
t0 = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(600)
    print(f"--- {(time.time()-t0)/3600:.1f} h")
    for g in range(n):
        lines = [l for l in open(f"/kaggle/working/gpu{g}.log").read().splitlines() if "round=" in l or "Dropping" in l or "Error" in l]
        print(f"gpu{g}:", lines[-2:] if lines else "starting")
for g in range(n):
    log = open(f"/kaggle/working/gpu{g}.log").read()
    if "Traceback" in log: print(f"!!! error on gpu{g}:", log[-3000:])
print("exit codes:", [p.returncode for p in procs])

In [ ]:
# check: only 'unclear' (disposal) may be dropped; anything else means a label mismatch
!grep -h "Dropping" /kaggle/working/gpu*.log | sort | uniq -c
# summary (paper_report.py needs all three tasks, so the table is built here instead)
import pandas as pd
r = pd.concat([pd.read_csv(f) for f in glob.glob(f"{OUT}/*/seed_*/per_court_test.csv")])
per_seed = r.groupby(["task", "seed"]).macro_f1.agg(mean_over_courts="mean", worst_court="min")
print(per_seed.round(4))
print("\nmean ± std over seeds:")
print(per_seed.groupby("task").agg(["mean", "std"]).round(4))
print("\nper court (mean over seeds):")
print(r.groupby(["task", "court"]).macro_f1.mean().unstack(0).round(3))
r.to_csv(f"{OUT}/court_results.csv", index=False); per_seed.to_csv(f"{OUT}/summary_per_seed.csv")